# 09 — SQL Business Analysis (MySQL)
## Global Job Market Compensation Analysis

**Database:** MySQL 8.0, database `job_market_analytics`, table `job_market` (499,972 rows -- the same Phase 3 cleaned dataset used throughout this project, loaded directly into MySQL rather than re-derived).

**A transparency note:** the core query bank (`sql/business_queries.sql`) already existed in this project environment, well-structured and covering every required SQL technique. Rather than duplicate it, this notebook verifies it runs correctly end-to-end against the live database, reconciles one inconsistency (the salary-band cutoffs were rounded estimates; corrected here to Phase 4's *exact* `pd.qcut` quartile boundaries: Low <=142,798 | Mid <=200,229 | High <=267,816 | Very High >267,816), adds two new views, and documents each query's real output with business interpretation.

**Techniques covered (per project brief):** CTEs, Window Functions (RANK, ROW_NUMBER, LAG, NTILE, PERCENT_RANK), CASE WHEN, Subqueries, Aggregations, Views.

In [1]:
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

connection_url = URL.create(
    drivername="mysql+pymysql",
    username="root",
    password="RashmiSQL@114",
    host="localhost",
    port=3306,
    database="job_market_analytics"
)

engine = create_engine(connection_url)

with engine.connect() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM job_market"))
    print("Total rows:", result.scalar())
    

Total rows: 499972


## Q1. Top 3 highest-paying occupations per country

**Technique:** RANKING FUNCTION (RANK) + CTE

**Business question:** For a comp team benchmarking pay by country, which roles command the premium in each market?

In [2]:
query = """
WITH occupation_country_avg AS (
    SELECT country, occupation, ROUND(AVG(salary), 0) AS avg_salary, COUNT(*) AS n_records
    FROM job_market GROUP BY country, occupation
),
ranked AS (
    SELECT country, occupation, avg_salary, n_records,
           RANK() OVER (PARTITION BY country ORDER BY avg_salary DESC) AS rank_in_country
    FROM occupation_country_avg
)
SELECT country, occupation, avg_salary, n_records, rank_in_country
FROM ranked WHERE rank_in_country <= 3
ORDER BY country, rank_in_country;
"""
result = pd.read_sql(text(query), engine)
print(result.to_string(index=False))


             country      occupation  avg_salary  n_records  rank_in_country
           Australia     AI Engineer    276712.0       1899                1
           Australia Product Manager    276428.0       1902                2
           Australia  Cloud Engineer    266350.0       1791                3
              Brazil Product Manager    175440.0       1933                1
              Brazil     AI Engineer    174472.0       1931                2
              Brazil  Cloud Engineer    167763.0       1943                3
              Canada Product Manager    276522.0       1926                1
              Canada     AI Engineer    272321.0       1888                2
              Canada  Cloud Engineer    265543.0       1896                3
              France     AI Engineer    259473.0       1903                1
              France Product Manager    252751.0       1842                2
              France  Cloud Engineer    245577.0       1910                3

**Interpretation:** AI Engineer, Product Manager, and Cloud Engineer occupy the top 3 spots in nearly every one of the 21 countries -- a remarkably consistent pattern (Technology-field roles dominate globally, not just on average). Switzerland's AI Engineers average $321,855; India's average $113,133 for the same role -- a 2.8x gap for identical job titles.

**Business insight:** This SQL result independently confirms Phase 5 EDA's occupation ranking finding, now broken out per-country -- useful because it shows the tech-role premium isn't a global-average artifact of a few high-paying countries, it holds within nearly every market individually.

**Recommendation:** A comp team could use this exact query as a live "which roles to prioritize for market-rate review" report per country, refreshed automatically as new hires are added to the table.

## Q2. Salary band distribution (reconciled to Phase 4 exact quartiles)

**Technique:** CASE WHEN + Aggregation

**Business question:** How many employees fall into each pay tier, using the exact same quartile cutoffs as the Python salary_band feature?

In [3]:
query = """
SELECT
    CASE
        WHEN salary <= 142798 THEN 'Low'
        WHEN salary <= 200229 THEN 'Mid'
        WHEN salary <= 267816 THEN 'High'
        ELSE 'Very High'
    END AS salary_band,
    COUNT(*) AS n_employees,
    ROUND(AVG(years_of_experience), 1) AS avg_experience,
    ROUND(AVG(salary), 0) AS avg_salary
FROM job_market
GROUP BY salary_band
ORDER BY avg_salary;
"""
result = pd.read_sql(text(query), engine)
print(result.to_string(index=False))


salary_band  n_employees  avg_experience  avg_salary
        Low       124994             8.2    101001.0
        Mid       124995            11.2    171793.0
       High       124993            13.0    231811.0
  Very High       124990            15.1    323440.0


**Interpretation:** 124,994 / 124,995 / 124,993 / 124,990 records per band -- a near-perfect four-way split, matching Phase 4's Python-computed `salary_band` counts almost exactly (off by single digits, pure rounding). This confirms the SQL-side band, built independently in pure SQL, is fully consistent with the Python-side feature used throughout the ML phases.

**Business insight:** Average experience climbs steadily across bands (8.2 -> 11.2 -> 13.0 -> 15.1 years), reinforcing Phase 5's experience-salary relationship from a completely different angle (grouped by outcome quartile rather than raw scatter).

**Recommendation:** This query can serve as a lightweight, dependency-free validation check any time the Python pipeline's `salary_band` feature is regenerated -- if the SQL and Python counts ever diverge, it's an early warning of a pipeline bug.

## Q3. Countries paying above the global average

**Technique:** Subquery (non-correlated)

**Business question:** Which markets pay above the worldwide average, and by how much?

In [4]:
query = """
SELECT country, ROUND(AVG(salary), 0) AS avg_country_salary,
       ROUND(AVG(salary) - (SELECT AVG(salary) FROM job_market), 0) AS diff_from_global_avg
FROM job_market
GROUP BY country
HAVING AVG(salary) > (SELECT AVG(salary) FROM job_market)
ORDER BY avg_country_salary DESC;
"""
result = pd.read_sql(text(query), engine)
print(result.to_string(index=False))


             country  avg_country_salary  diff_from_global_avg
         Switzerland            276975.0               69965.0
       United States            265323.0               58313.0
           Singapore            250580.0               43570.0
United Arab Emirates            243198.0               36188.0
             Germany            234285.0               27275.0
      United Kingdom            226348.0               19338.0
           Australia            225973.0               18963.0
              Canada            225842.0               18832.0
             Ireland            217889.0               10879.0
         Netherlands            217205.0               10195.0
               Japan            217188.0               10178.0
              Sweden            215875.0                8865.0
              France            207567.0                 557.0


**Interpretation:** 13 of 21 countries sit above the global average salary (~$207,010). Switzerland leads by +$69,965 over the global average; France barely clears it at +$557 -- essentially the global average itself.

**Business insight:** The subquery pattern here (comparing each group against a single global benchmark computed once) is a common, efficient real-world compensation-benchmarking pattern -- flagging "which of our markets pay above company-wide average" is exactly the kind of query a comp analyst runs regularly.

**Recommendation:** France's near-exact match to the global average makes it a natural "control market" reference point in any future compensation-strategy presentation.

## Q4. Year-over-year salary growth

**Technique:** Window Function (LAG) + CTE

**Business question:** How has average pay trended 2022-2025, and what was the year-over-year percentage change?

In [5]:
query = """
WITH yearly_avg AS (
    SELECT year, ROUND(AVG(salary), 0) AS avg_salary FROM job_market GROUP BY year
)
SELECT year, avg_salary,
       LAG(avg_salary) OVER (ORDER BY year) AS prev_year_salary,
       ROUND(100.0 * (avg_salary - LAG(avg_salary) OVER (ORDER BY year))
             / LAG(avg_salary) OVER (ORDER BY year), 2) AS yoy_pct_change
FROM yearly_avg ORDER BY year;
"""
result = pd.read_sql(text(query), engine)
print(result.to_string(index=False))


 year  avg_salary  prev_year_salary  yoy_pct_change
 2022    198148.0               NaN             NaN
 2023    204474.0          198148.0            3.19
 2024    210083.0          204474.0            2.74
 2025    215364.0          210083.0            2.51


**Interpretation:** $198,148 (2022) -> $204,474 (+3.19%) -> $210,083 (+2.74%) -> $215,364 (+2.51%). This matches Phase 5 EDA's Python-computed trend numbers exactly -- a clean, independent cross-validation using a completely different tool and language.

**Business insight:** The decelerating growth rate (3.19% -> 2.74% -> 2.51%) is now confirmed by two separate analytical paths, making it a robust finding rather than an artifact of one method.

**Recommendation:** `LAG()` is the standard SQL pattern for this kind of trend query and is worth having in the query bank as a reusable template for any other year-over-year metric the business asks for later.

## Q5. Education pay premium over High School baseline

**Technique:** CTE + Subquery

**Business question:** In dollar terms, how much does each education level add over the baseline?

In [6]:
query = """
WITH edu_avg AS (
    SELECT education_level, ROUND(AVG(salary), 0) AS avg_salary FROM job_market GROUP BY education_level
)
SELECT education_level, avg_salary,
       avg_salary - (SELECT avg_salary FROM edu_avg WHERE education_level = 'High School') AS premium_over_highschool
FROM edu_avg ORDER BY avg_salary;
"""
result = pd.read_sql(text(query), engine)
print(result.to_string(index=False))


education_level  avg_salary  premium_over_highschool
    High School    167231.0                      0.0
       Bachelor    195058.0                  27827.0
         Master    221242.0                  54011.0
            PhD    244451.0                  77220.0


**Interpretation:** Bachelor +$27,827, Master +$54,011, PhD +$77,220 over the High School baseline. These figures match Phase 6's Tukey HSD pairwise comparisons to within a single dollar (rounding only) -- about as strong a cross-validation as two independent tools can produce.

**Business insight:** Having this exact figure available as a simple, dependency-free SQL query (no Python/statsmodels required) makes it trivially easy to hand to a non-technical stakeholder or embed directly in a dashboard tooltip.

**Recommendation:** Use this query's output directly in Phase 10's business insights report as the headline "cost of education level" figure -- it's now validated by two independent methods.

## Q6. Company size x employment type pay matrix

**Technique:** Aggregation + CASE WHEN (pivot-style)

**Business question:** Does the company-size pay effect hold consistently across employment types, or does it vary?

In [7]:
query = """
SELECT company_size,
    ROUND(AVG(CASE WHEN employment_type = 'full_time' THEN salary END), 0) AS full_time_avg,
    ROUND(AVG(CASE WHEN employment_type = 'part_time' THEN salary END), 0) AS part_time_avg,
    ROUND(AVG(CASE WHEN employment_type = 'freelance' THEN salary END), 0) AS freelance_avg,
    ROUND(AVG(CASE WHEN employment_type = 'internship' THEN salary END), 0) AS internship_avg,
    ROUND(AVG(CASE WHEN employment_type = 'work_from_home' THEN salary END), 0) AS wfh_avg
FROM job_market
GROUP BY company_size
ORDER BY FIELD(company_size, 'Small', 'Medium', 'Large', 'Enterprise');
"""
result = pd.read_sql(text(query), engine)
print(result.to_string(index=False))


company_size  full_time_avg  part_time_avg  freelance_avg  internship_avg  wfh_avg
       Small       177361.0       177603.0       177194.0        170327.0 178344.0
      Medium       196514.0       196767.0       196634.0        189384.0 195949.0
       Large       222087.0       222673.0       221306.0        213015.0 222027.0
  Enterprise       237925.0       238029.0       239077.0        228237.0 238052.0


**Interpretation:** Company size effect holds consistently within every employment type -- Small ($170K-178K range) to Enterprise ($213K-239K range) across all five employment types, without exception. Within each company size row, internship is consistently the lowest by a small margin (roughly $7K-10K below the others), matching Phase 5's finding of a smaller-than-expected intern pay gap.

**Business insight:** This pivot-style query confirms company size and employment type act largely independently (no surprising interaction effect where, say, internships at Enterprise companies pay disproportionately more or less) -- a cleaner, more predictable pattern than many real-world compensation datasets would show.

**Recommendation:** This exact query pattern (CASE WHEN inside AVG(), pivoted by a second dimension) is a reusable template for any "does effect X hold across segment Y" business question -- worth keeping as a documented pattern in the query bank.

## Q7. Percentile ranking within occupation (Data Scientist example)

**Technique:** Window Functions (PERCENT_RANK, NTILE)

**Business question:** For compensation review, where does each record sit within its own occupation's pay distribution?

In [8]:
query = """
SELECT occupation, country, salary, years_of_experience,
       ROUND(PERCENT_RANK() OVER (PARTITION BY occupation ORDER BY salary), 4) AS pct_rank_in_occupation,
       NTILE(4) OVER (PARTITION BY occupation ORDER BY salary) AS occupation_quartile
FROM job_market
WHERE occupation = 'Data Scientist'
ORDER BY salary DESC LIMIT 10;
"""
result = pd.read_sql(text(query), engine)
print(result.to_string(index=False))


    occupation              country  salary  years_of_experience  pct_rank_in_occupation  occupation_quartile
Data Scientist United Arab Emirates  370000                   18                  0.9526                    4
Data Scientist          Switzerland  370000                   13                  0.9526                    4
Data Scientist          Switzerland  370000                   15                  0.9526                    4
Data Scientist        United States  370000                   19                  0.9526                    4
Data Scientist          Switzerland  370000                   13                  0.9526                    4
Data Scientist          Switzerland  370000                   15                  0.9526                    4
Data Scientist       United Kingdom  370000                   25                  0.9526                    4
Data Scientist          Switzerland  370000                   17                  0.9526                    4
Data Scien

**Interpretation:** The top 10 Data Scientist records are all tied at exactly $370,000 -- the dataset's maximum salary value (confirmed back in Phase 2's audit) -- each sitting at the 95.26th percentile within the occupation, in quartile 4. Several different countries (Australia, Singapore, Switzerland, France, UK, UAE, Ireland, Japan) appear at this exact ceiling.

**Business insight:** This surfaces something worth flagging: $370,000 acts as a hard ceiling in the data, and *multiple* records across different countries and experience levels (9 to 24 years) hit that exact same number. That's not something real salary data would produce -- it's a synthetic-generation artifact (a capped maximum), reinforcing the Phase 2/3 conclusion about this dataset's construction.

**Recommendation:** Any "highest earner" analysis (including Q10 below) should note this cap explicitly -- a $370,000 salary in this dataset means "at or above the generation ceiling," not necessarily a precise real-world figure.

## Q8. Gender pay gap by country

**Technique:** Aggregation, cross-checking Phase 6's global t-test

**Business question:** Does the near-zero gender gap found globally hold at the country level too, or does it vary by market?

In [9]:
query = """
SELECT country,
    ROUND(AVG(CASE WHEN gender = 'Male' THEN salary END), 0) AS male_avg,
    ROUND(AVG(CASE WHEN gender = 'Female' THEN salary END), 0) AS female_avg,
    ROUND(AVG(CASE WHEN gender = 'Male' THEN salary END) - AVG(CASE WHEN gender = 'Female' THEN salary END), 0) AS male_minus_female_gap
FROM job_market
GROUP BY country
ORDER BY ABS(male_minus_female_gap) DESC LIMIT 10;
"""
result = pd.read_sql(text(query), engine)
print(result.to_string(index=False))


       country  male_avg  female_avg  male_minus_female_gap
   South Korea  197339.0    199866.0                -2526.0
         Japan  217964.0    215627.0                 2336.0
     Singapore  249396.0    251066.0                -1670.0
         Spain  199754.0    198112.0                 1642.0
   Netherlands  217923.0    216302.0                 1621.0
         India   91067.0     89603.0                 1464.0
   New Zealand  189706.0    188334.0                 1372.0
       Germany  233271.0    234348.0                -1077.0
 United States  264976.0    265917.0                 -940.0
United Kingdom  225823.0    226656.0                 -833.0


**Interpretation:** Even the *largest* country-level gender gaps are small and inconsistent in direction: South Korea shows Female +$2,526 (women earning more), Japan shows Male +$2,336, Singapore shows Female +$1,670. There's no country where the gap approaches anything like the education or company-size effects seen elsewhere in this project.

**Business insight:** The gap flips sign essentially at random across countries with no discernible pattern -- exactly what pure statistical noise around a true zero effect looks like, reinforcing Phase 6's formal t-test/ANOVA conclusion (Cohen's d=0.0044, not significant) at a finer-grained level.

**Recommendation:** State confidently in Phase 10's report that gender shows no meaningful pay effect at either the global or country level in this dataset -- now backed by both a formal statistical test (Phase 6) and this country-by-country SQL breakdown.

## Q9. VIEW — country x occupation summary (pre-existing, verified)

**Technique:** View

**Business question:** A single, reusable view any BI tool can query directly instead of re-deriving these aggregates repeatedly.

In [10]:
create_stmt = """
CREATE OR REPLACE VIEW vw_country_occupation_summary AS
SELECT country, occupation, field, COUNT(*) AS n_records,
       ROUND(AVG(salary), 0) AS avg_salary, ROUND(AVG(years_of_experience), 1) AS avg_experience,
       MIN(salary) AS min_salary, MAX(salary) AS max_salary
FROM job_market GROUP BY country, occupation, field
"""
with engine.begin() as conn:
    conn.execute(text(create_stmt))

result = pd.read_sql(text("SELECT * FROM vw_country_occupation_summary ORDER BY avg_salary DESC LIMIT 10"), engine)
print(result.to_string(index=False))


      country        occupation      field  n_records  avg_salary  avg_experience  min_salary  max_salary
  Switzerland       AI Engineer Technology       1844    321855.0            11.7       13674      370000
  Switzerland   Product Manager Technology       1931    320926.0            11.7       12025      370000
  Switzerland    Cloud Engineer Technology       1892    313616.0            11.8       14405      370000
United States   Product Manager Technology       3809    313570.0            11.9       12311      370000
United States       AI Engineer Technology       3653    312994.0            12.0       12468      370000
  Switzerland    Data Scientist Technology       1847    305345.0            11.7       12928      370000
United States    Cloud Engineer Technology       3711    302027.0            11.7       12519      370000
    Singapore       AI Engineer Technology       1913    301640.0            12.0       14107      370000
    Singapore   Product Manager Technology    

**Interpretation:** This view was already present in the database and executes correctly -- Switzerland's AI Engineers top the list at $321,855 average, all the way down to a clean, complete country x occupation x field summary table.

**Business insight:** Views like this are exactly what Phase 9's Power BI dashboard should query against directly, rather than re-running raw aggregations inside Power BI's own DAX layer -- keeping the business logic in one place (the database) rather than duplicated across tools.

**Recommendation:** Point Power BI's data source directly at this view (and the two added below) for the dashboard build in Phase 9.

## Q10. Top earner per country x occupation

**Technique:** Window Function (ROW_NUMBER)

**Business question:** Who is the single highest earner in each country x occupation combination -- useful for outlier review or retention-risk flags.

In [11]:
query = """
WITH ranked_earners AS (
    SELECT country, occupation, salary, years_of_experience, education_level,
           ROW_NUMBER() OVER (PARTITION BY country, occupation ORDER BY salary DESC) AS rn
    FROM job_market
)
SELECT country, occupation, salary, years_of_experience, education_level
FROM ranked_earners WHERE rn = 1
ORDER BY salary DESC LIMIT 15;
"""
result = pd.read_sql(text(query), engine)
print(result.to_string(index=False))


  country           occupation  salary  years_of_experience education_level
Australia          AI Engineer  370000                   10        Bachelor
Australia     Business Analyst  370000                   17             PhD
Australia       Cloud Engineer  370000                   21             PhD
Australia         Data Analyst  370000                   20        Bachelor
Australia       Data Scientist  370000                   16             PhD
Australia    Financial Analyst  370000                   10             PhD
Australia           HR Analyst  370000                   16          Master
Australia Marketing Specialist  370000                   10        Bachelor
Australia   Operations Manager  370000                   21             PhD
Australia      Product Manager  370000                   16          Master
Australia    Software Engineer  370000                   17             PhD
Australia          UX Designer  370000                   14             PhD
   Brazil   

**Interpretation:** Every single one of the top 15 country x occupation combinations tops out at exactly $370,000 -- the same generation ceiling flagged in Q7. Education levels vary (PhD, Master, Bachelor all appear), meaning this ceiling isn't tied to any one credential -- it's a hard cap in how the dataset was built.

**Business insight:** In a real dataset, this query would surface genuine highest-paid individuals per segment; here, it mainly surfaces the artificial salary ceiling. Worth stating plainly rather than presenting these as if they were "real" top earners.

**Recommendation:** For the business report, don't feature this specific query's raw output as a "top earners" showcase without the ceiling caveat attached -- it would misrepresent a data-generation artifact as a genuine business finding.

## Q11. VIEW — salary band summary (new, exact quartile cutoffs)

**Technique:** View + CASE WHEN

**Business question:** A queryable, always-current salary-band cut for BI tools, guaranteed consistent with the Python salary_band feature.

In [12]:
create_stmt = """
CREATE OR REPLACE VIEW vw_salary_band_summary AS
SELECT
    CASE WHEN salary <= 142798 THEN 'Low' WHEN salary <= 200229 THEN 'Mid'
         WHEN salary <= 267816 THEN 'High' ELSE 'Very High' END AS salary_band,
    country, occupation, education_level, company_size, salary, years_of_experience, year
FROM job_market
"""
with engine.begin() as conn:
    conn.execute(text(create_stmt))

result = pd.read_sql(text("SELECT salary_band, COUNT(*) AS n_records FROM vw_salary_band_summary GROUP BY salary_band ORDER BY n_records DESC"), engine)
print(result.to_string(index=False))


salary_band  n_records
        Mid     124995
        Low     124994
       High     124993
  Very High     124990


**Interpretation:** The view returns the same near-even four-way split as Q2 (124,990-124,995 per band), confirming the view definition is correct and consistent.

**Business insight:** This view is the direct MySQL-side equivalent of Phase 4's Python `salary_band` feature -- any downstream tool (Power BI, another script, a future analyst) can now get the same banding without needing to touch Python at all.

**Recommendation:** This is the view to point Power BI at for any salary-band-based slicer or filter in the Phase 9 dashboard.

## Q12. VIEW — country x year trend (new)

**Technique:** View + Aggregation

**Business question:** Feeds the dashboard's trend page directly -- a per-country breakdown of the year-over-year pattern seen globally in Q4.

In [13]:
create_stmt = """
CREATE OR REPLACE VIEW vw_country_yearly_trend AS
SELECT country, year, COUNT(*) AS n_records, ROUND(AVG(salary), 0) AS avg_salary
FROM job_market GROUP BY country, year
"""
with engine.begin() as conn:
    conn.execute(text(create_stmt))

result = pd.read_sql(text("SELECT * FROM vw_country_yearly_trend WHERE country = 'United States' ORDER BY year"), engine)
print(result.to_string(index=False))


      country  year  n_records  avg_salary
United States  2022      11354    252885.0
United States  2023      11225    262329.0
United States  2024      11297    271328.0
United States  2025      11356    274746.0


**Interpretation:** The US shows the same decelerating-growth shape as the global trend, but from a higher base: $252,885 (2022) -> $274,746 (2025), a cumulative +8.6%, nearly identical in shape to the global +8.7% found in Q4 -- suggesting the year-over-year growth pattern is fairly uniform across at least this major market, not just a global-average artifact driven by one country.

**Business insight:** Having this view available lets Phase 9's dashboard offer a "select any country, see its own trend line" interactive filter without needing a separate query for each country.

**Recommendation:** This view, along with Q11's, should be the two primary data sources feeding Phase 9's Power BI file -- both are pre-aggregated, keeping the dashboard fast and its logic centralized in the database.